# Text processing and word representations / Xử lý văn bản và biểu diễn từ

Tôi lưu bài này trong nhật ký học để ôn lại cách viết và đọc code.

**Nguồn / Source:** Bản Colab `3248.ipynb`. Các bài có tên “Bản sao” là tài liệu thực hành tham khảo; tên file không được xem là bằng chứng tác giả.

**Cách dùng / Run:** chạy các cell theo thứ tự. Thư viện được dùng: `collections`, `nltk`, `nltk.tokenize`, `numpy`, `typing`.

**Execution:** ARCHIVED_SOURCE — bản mã học tập, chưa ghi nhận chạy lại đầy đủ trong đợt này. Output lịch sử được bỏ; hướng dẫn và lỗi cần xử lý nằm trong bảng kỹ thuật.

Learning archive: coursework/tutorial practice, with original code retained for study. Original tutorial references embedded in the source are retained below.

In [ ]:
from typing import List, Tuple, Dict
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
from collections import Counter

# Ensure necessary NLTK tokenizer models are available
nltk.download("punkt")
nltk.download('punkt_tab')

class Tokenizer:
    def __init__(self, tokenize_type: str = "basic", lowercase: bool = False):
        self.lowercase = lowercase
        self.type = tokenize_type
        self.vocab = []  # Empty vocabulary list

    def basicTokenize(self, string: str) -> List[str]:
        # Tokenizes input string by splitting on whitespace
        ### BEGIN SOLUTION
        return string.split()
        ### END SOLUTION

    def nltkTokenize(self, string: str) -> List[str]:
        # Tokenizes input string using NLTK's word tokenizer
        ### BEGIN SOLUTION
        return word_tokenize(string)
        ### END SOLUTION

    def tokenize(self, string: str) -> List[str]:
        # Tokenizes string and updates vocabulary with unique words
        if self.lowercase:
            string = string.lower()
        tokens = self.basicTokenize(string) if self.type == "basic" else self.nltkTokenize(string)
        self.vocab += [w for w in set(tokens) if w not in self.vocab]
        return tokens

    def countTopWords(self, words: List[str], k: int) -> List[Tuple[str, int]]:
        # Returns the top k most common words
        ### BEGIN SOLUTION
        counter = Counter(words)
        return counter.most_common(k)
        ### END SOLUTION

In [ ]:
class BiGramLanguageModel:
    def __init__(self, vocab: List[str], smoothing: str = None, smoothing_param: float = None): pass
    def computeBigramProb(self): pass
    def computeBigramProbAddAlpha(self, alpha: float = 0.001): pass
    def train(self, corpus: List[str]): pass
    def test(self, corpus: List[str]) -> float: pass

In [ ]:
def bigram_init(self, vocab, smoothing=None, smoothing_param=None):
    self.vocab = vocab
    self.token_to_idx = {word: i for i, word in enumerate(vocab)}
    self.smoothing = smoothing
    self.smoothing_param = smoothing_param
    self.bi_counts = None
    self.bi_prob = None
    assert smoothing is None or smoothing_param is not None, "Smoothing parameters must be set correctly."

In [ ]:
def bigram_compute_prob(self):
    ### BEGIN SOLUTION
    self.bi_prob = np.zeros_like(self.bi_counts, dtype=float)
    row_sums = self.bi_counts.sum(axis=1, keepdims=True)
    np.divide(self.bi_counts, row_sums, out=self.bi_prob, where=row_sums!=0)
    ### END SOLUTION

In [ ]:
def bigram_compute_prob_add_alpha(self, alpha=1e-3):
    ### BEGIN SOLUTION
    self.bi_prob = np.zeros_like(self.bi_counts, dtype=float)
    smoothed_counts = self.bi_counts + alpha
    row_sums = smoothed_counts.sum(axis=1, keepdims=True)
    np.divide(smoothed_counts, row_sums, out=self.bi_prob, where=row_sums!=0)
    ### END SOLUTION

In [ ]:
def bigram_train(self, corpus):
    self.bi_counts = np.zeros((len(self.vocab), len(self.vocab)), dtype=float)
    corpus_indices = [self.token_to_idx[w] for w in corpus]
    for i in range(len(corpus_indices) - 1):
        self.bi_counts[corpus_indices[i]][corpus_indices[i + 1]] += 1
    if self.smoothing == "addAlpha":
        self.computeBigramProbAddAlpha(self.smoothing_param)
    else:
        self.computeBigramProb()

In [ ]:
def bigram_test(self, corpus):
    logprob = 0.0
    eps = 1e-10
    corpus_indices = [self.token_to_idx[w] for w in corpus]
    for i in range(len(corpus_indices) - 1):
        prob = self.bi_prob[corpus_indices[i], corpus_indices[i + 1]]
        logprob += np.log(prob + eps)
    logprob /= len(corpus_indices) - 1
    return np.exp(-logprob)

In [ ]:
BiGramLanguageModel.__init__ = bigram_init
BiGramLanguageModel.computeBigramProb = bigram_compute_prob
BiGramLanguageModel.computeBigramProbAddAlpha = bigram_compute_prob_add_alpha
BiGramLanguageModel.train = bigram_train
BiGramLanguageModel.test = bigram_test

In [ ]:
def readCorpus(filename: str, tokenizer: Tokenizer) -> List[str]:
      # Reads and tokenizes the corpus from a file
    ### BEGIN SOLUTION
    try:
        with open(filename, 'r', encoding='utf-8') as f:
            text = f.read()
        return tokenizer.tokenize(text)
    except FileNotFoundError:
        print(f"Warning: File {filename} not found. Using sample text.")
        sample_text = "This is a sample text for language modeling. It contains multiple sentences to demonstrate the functionality of the tokenizer and bigram model."
        return tokenizer.tokenize(sample_text)

In [ ]:
def runLanguageModel(train_corpus: List[str], val_corpus: List[str], tokenizer: Tokenizer, smoothing_type: str = None, smoothing_param: float = 0.0) -> Dict[str, float]:
    # Trains and tests the language model, returning key metrics
    lm = BiGramLanguageModel(tokenizer.vocab, smoothing=smoothing_type, smoothing_param=smoothing_param)
    lm.train(train_corpus)
    return {"train_ppl": lm.test(train_corpus), "val_ppl": lm.test(val_corpus)}

In [ ]:
# Initialize tokenizers with basic and NLTK options, both set to lowercase.
basic_tokenizer = Tokenizer(tokenize_type='basic', lowercase=True)
nltk_tokenizer = Tokenizer(tokenize_type='nltk', lowercase=True)

In [ ]:
# Read and tokenize the training and validation corpora using the basic tokenizer.
train_corpus = readCorpus('./data/train.txt', basic_tokenizer)
val_corpus = readCorpus('./data/val.txt', basic_tokenizer)

# Example of using the NLTK tokenizer for comparison (unused in final results).
train_corpus_nltk = readCorpus('./data/train.txt', nltk_tokenizer)
val_corpus_nltk = readCorpus('./data/val.txt', nltk_tokenizer)

In [ ]:
# Get top 10 frequent words and counts from train_corpus with basic_tokenizer.
### BEGIN PUBLIC TESTS
basic_tokenizer.countTopWords(train_corpus, k=10)
### END PUBLIC TESTS

In [ ]:
# Get top 10 frequent words and counts from train_corpus_nltk with nltk_tokenizer.
### BEGIN PUBLIC TESTS
nltk_tokenizer.countTopWords(train_corpus_nltk, k=10)
### END PUBLIC TESTS

In [ ]:
# Run the language model with the basic tokenizer and without smoothing.
### BEGIN PUBLIC TESTS
runLanguageModel(train_corpus, val_corpus,
                 tokenizer=basic_tokenizer)
### END PUBLIC TESTS

In [ ]:
# Run the language model with the nltk tokenizer and without smoothing.
### BEGIN PUBLIC TESTS
runLanguageModel(train_corpus_nltk, val_corpus_nltk,
                 tokenizer=nltk_tokenizer)
### END PUBLIC TESTS

In [ ]:
# Run the language model with the basic tokenizer and with smoothing.
runLanguageModel(train_corpus, val_corpus,
                 tokenizer=basic_tokenizer, smoothing_type='addAlpha', smoothing_param=10e-5)

In [ ]:
# Run the language model with the nltk tokenizer and with smoothing.
runLanguageModel(train_corpus_nltk, val_corpus_nltk,
                 tokenizer=nltk_tokenizer, smoothing_type='addAlpha', smoothing_param=10e-5)

## Ghi chú học tập / Study prompts

- Đầu vào của từng hàm là gì, đầu ra có kiểu và kích thước nào?
- Thay đổi tham số hoặc dữ liệu nhỏ sẽ ảnh hưởng kết quả ra sao?
- Cell nào cần dữ liệu bên ngoài, thư viện hoặc thao tác nhập tay?

Explain the data flow, modify one example, and record the result when revisiting this exercise.